# SCX detailed example - Python
Run the cells in order in your existing Scanpy/AnnData kernel.
This example makes small data, saves and reads SCX, and checks metadata and TSV output.
It checks the fixed alpha.7 script's SHA-256 before loading it. Nothing is installed.
The [R example](detailed.R) uses the same values and IDs.

For a clean start, restart the kernel and Run All.
Each full run makes a new output folder and keeps earlier results.
To repeat a write cell, run step 2 first for a new folder.
The first run needs internet. Later runs can use checked local files offline.


## 1. Check your environment
Choose the kernel with AnnData, usually your Scanpy environment.
If a package is missing, switch environments. This example does not run pip.


In [2]:
import importlib.util
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Choose your Python 3.10+ Scanpy/AnnData kernel.")
needed = ["anndata", "numpy", "pandas", "scipy"]
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Switch to your analysis kernel; missing: " + ", ".join(missing))

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

print("Ready: existing AnnData / NumPy / pandas / SciPy environment. No packages installed.")


Ready: existing AnnData / NumPy / pandas / SciPy environment. No packages installed.


## 2. Download and check the files
The hashes are fixed to v0.1.0-alpha.7, including SHA256SUMS itself.
Existing files are checked, not replaced.
For offline use, put scx.py and SHA256SUMS from that release in
`scx-tutorial/files/` under your working folder.
Each run makes a new `scx-tutorial/output/run-*` folder. Earlier files are kept.


In [3]:
from pathlib import Path
import hashlib
import tempfile
import time
import urllib.request

RELEASE = "v0.1.0-alpha.7"
BASE_URL = f"https://github.com/junha-shin/scx-format/releases/download/{RELEASE}/"
EXPECTED = {
    "scx.py": "3384f730e621c2ede7110da6b9e9c39bafb578fa886e8a9b752751c692b7fc92",
    "SHA256SUMS": "f7c277c380d04ed1f2766b0c9726d6b32c69811cf51a7b426808efb9cadfe914",
}
HOME = Path.cwd() / "scx-tutorial"
FILES = HOME / "files"
FILES.mkdir(parents=True, exist_ok=True)

def verified_file(name):
    path = FILES / name
    if path.exists():
        payload = path.read_bytes()
    else:
        for attempt in range(2):
            try:
                with urllib.request.urlopen(BASE_URL + name, timeout=20) as response:
                    payload = response.read()
                break
            except OSError as error:
                if attempt == 1:
                    raise RuntimeError("Download failed. Copy the alpha.7 files into scx-tutorial/files/ for offline use.") from error
                time.sleep(0.5)
    if hashlib.sha256(payload).hexdigest() != EXPECTED[name]:
        raise RuntimeError(f"Checksum mismatch: {name}. Inspect the cached file and download the pinned release again; it was not executed or overwritten.")
    if not path.exists():
        with path.open("xb") as stream:
            stream.write(payload)
    return path.resolve()

checksum_path = verified_file("SHA256SUMS")
SCRIPT_PATH = verified_file("scx.py")
assert EXPECTED["scx.py"] + "  scx.py" in checksum_path.read_text(encoding="utf-8").splitlines()

output_root = HOME / "output"
output_root.mkdir(exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=output_root))
print("Verified alpha.7 bytes. Created a fresh run-* folder under scx-tutorial/output/.")


Verified alpha.7 bytes. Created a fresh run-* folder under scx-tutorial/output/.


## 3. Load the checked file
Load the saved file directly to avoid an installed scx package.
Use the normal API through the variable `scx`.


In [4]:
loaded = sys.modules.get("_scx_standalone")
if loaded is not None and not str(getattr(loaded, "__file__", "")).startswith(str(SCRIPT_PATH) + "!/"):
    raise RuntimeError("Another standalone SCX is already loaded. Restart the kernel and Run All.")
spec = importlib.util.spec_from_file_location("scx_tutorial_verified", SCRIPT_PATH)
scx = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scx)
assert scx.SCX_VERSION == RELEASE
print("Loaded", scx.SCX_VERSION, "from the verified file.")


Loaded v0.1.0-alpha.7 from the verified file.


## 4. Make small data
Rows are cells; columns are features. This display has only 12 values.
Do not make a real dataset dense just to view it.
Here normalized values are `log1p(counts)`, not a biological analysis.
Metadata include numbers, boolean values, ordered categories, missing values,
and an unused category.


In [5]:
cell_ids = ["cell1", "cell2", "cell3"]
feature_ids = ["geneA", "geneB", "geneC", "geneD"]
counts = sparse.csr_matrix([[1, 0, 3, 0], [0, 2, 0, 4], [5, 1, 0, 0]], dtype=np.int32)
normalized = counts.astype(np.float64)
normalized.data = np.log1p(normalized.data)
obs = pd.DataFrame(index=cell_ids)
obs["sample_number"] = pd.array([1, None, 3], dtype="Int32")
obs["flag"] = pd.array([True, None, False], dtype="boolean")
obs["group"] = pd.Categorical(["B", "A", "B"], categories=["B", "A", "unused"], ordered=True)
var = pd.DataFrame({"symbol": pd.Series(["A", "B", "C", "D"], index=feature_ids)})
demo = ad.AnnData(X=normalized, obs=obs, var=var)
demo.layers["counts"] = counts

print("Expected: 3 cells × 4 features")
display(pd.DataFrame(counts.toarray(), index=cell_ids, columns=feature_ids))
display(demo.obs)


Expected: 3 cells × 4 features


,geneA,geneB,geneC,geneD
cell1,1,0,3,0
cell2,0,2,0,4
cell3,5,1,0,0


,sample_number,flag,group
cell1,1,True,B
cell2,<NA>,<NA>,A
cell3,3,False,B


## 5. Save, read, and check
Defaults read counts from `layers['counts']` and normalized values from `X`.
SCX checks the data automatically. Use `analysis=True` for embeddings and graphs
in a real export; this example has none.
Counts must match exactly. Normalized values must agree within `1e-12`.
Metadata checks include types, values, missing values, and category order.


In [6]:
scx_dir = RUN_DIR / "demo.scx"
scx.write_scx(demo, scx_dir)
restored = scx.read_scx(scx_dir)
np.testing.assert_array_equal(restored.layers["counts"].toarray(), counts.toarray())
normalized_error = float(np.max(np.abs(restored.X.toarray() - normalized.toarray())))
assert normalized_error <= 1e-12
pd.testing.assert_frame_equal(demo.obs, restored.obs, check_exact=True)
pd.testing.assert_frame_equal(demo.var, restored.var, check_exact=True)
assert restored.shape == (3, 4)
print("PASS: counts, normalized expression, both axes and typed metadata preserved.")
display(restored.obs)


PASS: counts, normalized expression, both axes and typed metadata preserved.


,sample_number,flag,group
cell1,1,True,B
cell2,<NA>,<NA>,A
cell3,3,False,B


## 6. View TSV output
The first column is cell_id. Feature columns include zeros.
Rows are cells; columns are features. Add .gz to compress the file.
The tables should match the counts and log1p values above.


In [7]:
counts_tsv = RUN_DIR / "counts.tsv"
normalized_tsv = RUN_DIR / "normalized.tsv.gz"
scx.to_tsv(scx_dir, counts_tsv)
scx.to_tsv(scx_dir, normalized_tsv, matrix="normalized")
counts_table = pd.read_csv(counts_tsv, sep="\t", index_col="cell_id")
normalized_table = pd.read_csv(normalized_tsv, sep="\t", index_col="cell_id")
np.testing.assert_array_equal(counts_table.to_numpy(), counts.toarray())
np.testing.assert_allclose(normalized_table.to_numpy(), normalized.toarray(), rtol=0, atol=1e-12)
display(counts_table)
display(normalized_table)


,geneA,geneB,geneC,geneD
cell_id,,,,
cell1,1,0,3,0
cell2,0,2,0,4
cell3,5,1,0,0


,geneA,geneB,geneC,geneD
cell_id,,,,
cell1,0.693147,0.000000,1.386294,0.000000
cell2,0.000000,1.098612,0.000000,1.609438
cell3,1.791759,0.693147,0.000000,0.000000


## 7. Select and order IDs
Select cell3, cell1 and geneB, geneA. Expect `[[1, 5], [0, 1]]`.
Use ID lists, not row numbers. Python and R keep the same requested order.


In [8]:
selected_cells = ["cell3", "cell1"]
selected_features = ["geneB", "geneA"]
subset_tsv = RUN_DIR / "subset.tsv.gz"
scx.to_tsv(scx_dir, subset_tsv, cells=selected_cells, features=selected_features)
subset_table = pd.read_csv(subset_tsv, sep="\t", index_col="cell_id")
assert subset_table.index.tolist() == selected_cells
assert subset_table.columns.tolist() == selected_features
np.testing.assert_array_equal(subset_table.to_numpy(), [[1, 5], [0, 1]])
display(subset_table)


,geneB,geneA
cell_id,,
cell3,1,5
cell1,0,1


## 8. Check existing output protection
This cell tries to write counts.tsv again and catches the expected error.
The file's bytes must stay unchanged.
Choose a new filename or run step 2. The example does not delete old output.


In [9]:
previous_hash = hashlib.sha256(counts_tsv.read_bytes()).hexdigest()
try:
    scx.to_tsv(scx_dir, counts_tsv)
except FileExistsError:
    print("Expected refusal: counts.tsv already exists; its bytes are preserved.")
else:
    raise AssertionError("Overwrite protection did not refuse the existing file")
assert hashlib.sha256(counts_tsv.read_bytes()).hexdigest() == previous_hash


Expected refusal: counts.tsv already exists; its bytes are preserved.


## 9. Try your own H5AD
Leave the switch False on the first run. Then set your path and raw-count source.
Use COUNTS_SOURCE="X" and NORMALIZED_SOURCE=None only if X holds raw counts.
SCX does not assume raw.X holds counts.
The input is read, not changed. Output goes into the new folder.
Only a small ID-based TSV preview is exported. Keep your original object.


In [10]:
USE_OWN_DATA = False
INPUT_H5AD = Path("my-data.h5ad")
COUNTS_SOURCE = "counts"
NORMALIZED_SOURCE = "X"

if USE_OWN_DATA:
    own = ad.read_h5ad(INPUT_H5AD)
    print("Your object:", own.shape, "layers:", list(own.layers.keys()))
    own_scx = RUN_DIR / "own.scx"
    scx.write_scx(own, own_scx, counts=COUNTS_SOURCE, normalized=NORMALIZED_SOURCE)
    own_restored = scx.read_scx(own_scx)
    scx.to_tsv(own_scx, RUN_DIR / "own-preview.tsv.gz",
               cells=own_restored.obs_names[:10].tolist(),
               features=own_restored.var_names[:10].tolist())
else:
    print("Own-data example skipped. Change the switch only when your path/counts choices are ready.")


Own-data example skipped. Change the switch only when your path/counts choices are ready.


## Results and next steps
The demo.scx folder holds the manifest, typed metadata, and sparse matrices.
The TSV files show expression values. Runs stay under `scx-tutorial/output/`.
Dense TSV can be large. Use subsets and gzip. Sparse input checks still need memory.

The [R file](detailed.R) writes new folders only and refuses overwrite=TRUE.
Both languages refuse existing TSV outputs.
[TSV guide](../docs/tsv-export.md) | [PBMC3k checks](../examples/tsv-pbmc3k/README.md).


In [11]:
tutorial_result = {
    "passed": True, "SCX_VERSION": scx.SCX_VERSION, "shape": list(restored.shape),
    "counts_equal": True, "normalized_max_absolute_error": normalized_error,
    "typed_metadata_equal": True, "TSV_equal": True, "subset_order_equal": True,
    "existing_output_preserved": True,
}
print("SUCCESS: all demo checks passed. No SCX package was installed.")
print("Files in this run:")
for item in sorted(RUN_DIR.rglob("*")):
    if item.is_file():
        print("  " + item.relative_to(RUN_DIR).as_posix())


SUCCESS: all demo checks passed. No SCX package was installed.
Files in this run:
  counts.tsv
  demo.scx/cells.tsv.gz
  demo.scx/features.tsv.gz
  demo.scx/manifest.tsv
  demo.scx/matrices/counts.mtx.gz
  demo.scx/matrices/normalized.mtx.gz
  demo.scx/metadata.schema.json
  normalized.tsv.gz
  subset.tsv.gz
